In [2]:
import spacy 
import numpy as np

nlp = spacy.load('en_core_web_sm')

def average_embedding(sentence):
    doc = nlp(sentence)
    vectors = [token.vector for token in doc if token.has_vector]

    return np.mean(vectors, axis=0) if vectors else np.zeros((nlp.vocab.vectors_length,))

sentence = 'the dog runs fast'
embedding =average_embedding(sentence)

print(embedding[:5])

[ 0.22581662 -0.11688708 -0.40350407 -0.08471677 -0.05002417]


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

#sample corpus
corpus = [
    'The dog runs fast',
    'A cat sleeps on the sofa',
    'Dogs and Cats are great pets'
]

#compute tf-idf scores

vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)
tfidf_scores = vectorizer.transform([sentence]).toarray()[0]
words = vectorizer.get_feature_names_out()

In [11]:
tfidf_scores

array([0.        , 0.        , 0.        , 0.        , 0.52863461,
       0.        , 0.52863461, 0.        , 0.        , 0.        ,
       0.52863461, 0.        , 0.        , 0.40204024])

In [12]:
words

array(['and', 'are', 'cat', 'cats', 'dog', 'dogs', 'fast', 'great', 'on',
       'pets', 'runs', 'sleeps', 'sofa', 'the'], dtype=object)

In [13]:
# Compute weighted embeddings
def tfidf_weighted_embedding(sentence):
    doc = nlp(sentence)
    word_embeddings = []
    weights = []

    for token in doc:
        word = token.text.lower()
        if word in words and token.has_vector:
            idx = list(words).index(word)
            word_embeddings.append(token.vector * tfidf_scores[idx])
            weights.append(tfidf_scores[idx])

    return np.sum(word_embeddings, axis=0) / np.sum(weights) if weights else np.zeros((nlp.vocab.vectors_length,))

embedding = tfidf_weighted_embedding(sentence)

print("Sentence Embedding (TF-IDF Weighted):", embedding[:5])  # Print first 5 values

Sentence Embedding (TF-IDF Weighted): [ 0.19531701 -0.09783764 -0.43420096 -0.13278251 -0.02624997]


In [14]:
from sklearn.decomposition import PCA

# Generate embeddings for all sentences in corpus
sentence_embeddings = np.array([average_embedding(sent) for sent in corpus])

# Apply PCA to reduce dimensions (e.g., from 300 to 50)
pca = PCA()
reduced_embeddings = pca.fit_transform(sentence_embeddings)

print("Reduced Sentence Embedding (PCA):", reduced_embeddings[0][:5])  # Print first 5 values

Reduced Sentence Embedding (PCA): [ 1.3942726e+00 -1.3193353e+00  1.6938222e-07]


In [ ]:
pip install sentence-transformers

In [15]:
from sentence_transformers import SentenceTransformer

# Load pre-trained SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Compute SBERT sentence embedding
sentence = "The dog runs fast"
embedding = model.encode(sentence)

print("Sentence Embedding (SBERT):", embedding[:5])  # Print first 5 values

ModuleNotFoundError: No module named 'sentence_transformers'